# ConvCNP Downscaling — Single-folder analysis

Detailed analysis of a single experiment folder. Set `ACTIVE_FOLDER`
in the configuration cell below, run from top to bottom, and the rest
of the notebook operates on results from that folder only.

For high-level cross-folder comparison, use `cross_folder_analysis.ipynb`.

This is the refactor of the previous `convcnp_experiment_analysis.ipynb`
that loaded everything from a single hardcoded `EXPERIMENT_DEFS` dict;
now it pulls metadata from `experiments/<folder>/experiments.yaml` via
`_helpers.py`.


In [ ]:
import sys
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import _helpers
import importlib
importlib.reload(_helpers)

warnings.filterwarnings('ignore', category=FutureWarning)

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.constrained_layout.use': True,
    'savefig.bbox': 'tight',
    'savefig.dpi': 200,
})


## 0. Configuration

Set `ACTIVE_FOLDER` to one of the available experiment folders below.
Re-run from this cell down to refresh the entire analysis.


In [ ]:
# All folders that have an experiments.yaml.
print('Available folders:')
for f in _helpers.list_folders():
    out_dir = _helpers.output_dir_for_folder(f)
    n_runs = len(list(out_dir.iterdir())) if out_dir.exists() else 0
    flag = '✓' if out_dir.exists() else '·'
    print(f'  {flag} {f:<32} ({n_runs} run dirs)')

# === EDIT THIS ===
ACTIVE_FOLDER = 'snapshot_14y_eu'
# =================

print(f'\nActive folder: {ACTIVE_FOLDER}')


## 1. Load results


In [ ]:
SEEDS = [42, 123, 456]

df = _helpers.load_folder_results(ACTIVE_FOLDER, seeds=SEEDS)
print(f'Loaded {len(df)} run results across {df["experiment"].nunique()} experiments')
print(f'Seeds per experiment:')
print(df.groupby('experiment')['seed'].count().to_string())


## 2. Summary tables

Identical layout to the previous notebook's per-variable summary
tables — `_helpers.print_summary` is the same function lifted out
verbatim.


In [ ]:
TARGET_VARIABLES = ['tmax', 'wind_mean', 't2m', 'wind']

for variable in TARGET_VARIABLES:
    col = f'{variable}_mae'
    if col not in df.columns or not df[col].notna().any():
        continue
    _helpers.print_summary(df, variable, title_suffix=ACTIVE_FOLDER)


## 3. Per-seed detail

Per-seed breakdown of every shortlist-eligible experiment. Useful
for spotting outlier seeds or training instabilities before averaging.


In [ ]:
for variable in TARGET_VARIABLES:
    mae_col = f'{variable}_mae'
    rmse_col = f'{variable}_rmse'
    if mae_col not in df.columns:
        continue
    mask = df[mae_col].notna()
    if not mask.any():
        continue

    vdf = df[mask].sort_values(['experiment', 'seed'])

    unit = '°C' if variable in ('tmax', 't2m') else 'm/s'
    print(f'\n{"=" * 100}')
    print(f'PER-SEED DETAIL — {variable}')
    print(f'{"=" * 100}')
    print(f'{"Experiment":<42} {"Seed":>5} {"Epoch":>6} {"MAE":>10} {"RMSE":>10} {"Val Loss":>10}')
    print(f'{"-" * 100}')

    prev_exp = None
    for _, row in vdf.iterrows():
        exp = row['experiment']
        if exp != prev_exp and prev_exp is not None:
            print()
        prev_exp = exp

        label = row['label']
        seed = row['seed']
        epoch = row.get('best_epoch', '?')
        mae = row[mae_col]
        rmse = row.get(rmse_col, float('nan'))
        val_loss = row.get('best_val_loss', float('nan'))

        # `x == x` is the standard NaN check, but None doesn't satisfy it
        # AND raises TypeError when format-string-applied — guard explicitly.
        def _fmt(v, prec):
            if v is None or v != v:
                return 'N/A'
            return f'{v:.{prec}f}'

        epoch_str = f'{epoch}' if epoch is not None and epoch == epoch else '?'
        mae_str = _fmt(mae, 3)
        rmse_str = _fmt(rmse, 3)
        loss_str = _fmt(val_loss, 4)

        print(f'{label:<42} {seed:>5} {epoch_str:>6} {mae_str:>10} {rmse_str:>10} {loss_str:>10}')


## 4. Shortlist + baseline resolution

Pick the top-N TESSERA-enhanced configs per target variable using the
same composite (MAE, MAE-stability) ranking as the original notebook.
The corresponding baseline is then resolved automatically via
`_helpers.BASELINE_NAMES`. The pair `(baseline, tessera_shortlist)`
drives every detail-analysis cell below.


In [ ]:
TOP_N = 3

# Resolve a (baseline, shortlist) pair for each variable that has data.
# Empty pairs are kept as `(baseline, [])` so downstream cells can skip
# silently rather than KeyError.
baselines: dict[str, str] = {}
shortlists: dict[str, list[str]] = {}

for var in TARGET_VARIABLES:
    col = f'{var}_mae'
    if col not in df.columns or not df[col].notna().any():
        continue
    if var not in _helpers.BASELINE_NAMES:
        continue
    raw = _helpers.shortlist_experiments(df, var, top_n=TOP_N, tessera_only=True, print_table=True)
    baseline, tessera_only = _helpers.get_baseline_and_shortlist(raw, var)
    baselines[var] = baseline
    shortlists[var] = tessera_only

# Convenient per-variable handles for the original cells below. These
# are deliberately named the same way as in the old notebook so the
# preserved detail-analysis cells work without further changes.
tmax_baseline = baselines.get('tmax', _helpers.BASELINE_NAMES['tmax'])
tmax_tessera_shortlist = shortlists.get('tmax', [])
wind_baseline = baselines.get('wind_mean', _helpers.BASELINE_NAMES['wind_mean'])
wind_tessera_shortlist = shortlists.get('wind_mean', [])
t2m_snap_baseline = baselines.get('t2m', _helpers.BASELINE_NAMES['t2m'])
t2m_snap_tessera_shortlist = shortlists.get('t2m', [])
wind_snap_baseline = baselines.get('wind', _helpers.BASELINE_NAMES['wind'])
wind_snap_tessera_shortlist = shortlists.get('wind', [])


## 5. Variant comparison


In [ ]:
def plot_comparison(df, variable, reference_mae=None, reference_label=None):
    """Bar chart comparing experiments for a single target variable."""
    mae_col = f'{variable}_mae'
    if mae_col not in df.columns:
        return
    mask = df[mae_col].notna()
    if not mask.any():
        return

    vdf = df[mask].copy()
    experiments = vdf['experiment'].unique()

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(experiments))
    for i, exp in enumerate(experiments):
        edata = vdf[vdf['experiment'] == exp]
        mean = edata[mae_col].mean()
        std = edata[mae_col].std() if len(edata) > 1 else 0
        colour = edata['colour'].iloc[0]
        label = edata['label'].iloc[0]
        ax.bar(i, mean, yerr=std, capsize=5, color=colour,
               edgecolor='black', linewidth=0.5, alpha=0.85, label=label)

    if reference_mae is not None:
        ax.axhline(reference_mae, color='red', linestyle='--', linewidth=1.5,
                   alpha=0.7, label=reference_label)

    ax.set_xticks(x)
    ax.set_xticklabels([vdf[vdf['experiment'] == e]['label'].iloc[0]
                        for e in experiments], rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Test MAE')
    unit = '°C' if variable in ('tmax', 't2m') else 'm/s'
    ax.set_title(f'{variable} — ConvCNP Experiment Comparison ({unit}) [{ACTIVE_FOLDER}]')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.show()


for variable in TARGET_VARIABLES:
    if variable not in baselines:
        continue
    all_exps = [baselines[variable]] + shortlists[variable]
    plot_comparison(df[df['experiment'].isin(all_exps)], variable)


## 6. Per-seed stability


In [ ]:
for variable in TARGET_VARIABLES:
    if variable not in baselines:
        continue
    all_exps = [baselines[variable]] + shortlists[variable]
    mae_col = f'{variable}_mae'
    rmse_col = f'{variable}_rmse'
    vdf = df[df['experiment'].isin(all_exps) & df[mae_col].notna()]
    if len(vdf) == 0:
        continue

    experiments = [e for e in all_exps if e in vdf['experiment'].values]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for i, exp in enumerate(experiments):
        edata = vdf[vdf['experiment'] == exp]
        colour = edata['colour'].iloc[0]
        label = edata['label'].iloc[0]
        jitter = np.random.uniform(-0.1, 0.1, len(edata))
        ax1.scatter(i + jitter, edata[mae_col], s=80, c=colour,
                    edgecolors='black', linewidth=0.5, label=label, zorder=3)
        ax2.scatter(i + jitter, edata[rmse_col], s=80, c=colour,
                    edgecolors='black', linewidth=0.5, zorder=3)

    for ax, metric_label in [(ax1, 'MAE'), (ax2, 'RMSE')]:
        ax.set_xticks(range(len(experiments)))
        ax.set_xticklabels([vdf[vdf['experiment'] == e]['label'].iloc[0][:25]
                            for e in experiments], rotation=25, ha='right', fontsize=8)
        unit = '°C' if variable in ('tmax', 't2m') else 'm/s'
        ax.set_ylabel(f'{metric_label} ({unit})')
        ax.grid(axis='y', alpha=0.3)

    ax1.legend(fontsize=7, loc='upper left')
    unit = '°C' if variable in ('tmax', 't2m') else 'm/s'
    plt.suptitle(f'{variable} — Per-Seed Results, Shortlisted ({unit})  [{ACTIVE_FOLDER}]', fontsize=13)
    plt.show()


## 7. Training curves


In [ ]:
def load_curves(run_dir):
    """Load training curves from a run directory."""
    p = Path(run_dir) / 'training_curves.npz'
    if not p.exists():
        return None
    return dict(np.load(p))


In [ ]:
for variable in TARGET_VARIABLES:
    if variable not in baselines:
        continue
    all_exps = [baselines[variable]] + shortlists[variable]
    mae_col = f'{variable}_mae'
    vdf = df[df['experiment'].isin(all_exps) & df[mae_col].notna()]
    if len(vdf) == 0:
        continue

    exps_with_curves = [e for e in all_exps if e in vdf['experiment'].values]
    if not exps_with_curves:
        continue

    n_exp = len(exps_with_curves)
    fig, axes = plt.subplots(1, n_exp, figsize=(5 * n_exp, 4), sharey=True)
    if n_exp == 1:
        axes = [axes]

    for ax, exp_name in zip(axes, exps_with_curves):
        edata = vdf[vdf['experiment'] == exp_name]
        label = edata['label'].iloc[0]
        for _, row in edata.iterrows():
            curves = load_curves(row['run_dir'])
            if curves is None or 'train_losses' not in curves:
                continue
            ax.plot(curves['train_losses'], alpha=0.7, label=f's{row["seed"]}')

        ax.set_title(label, fontsize=8)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    axes[0].set_ylabel('Train Loss (NLL)')
    plt.suptitle(f'{variable} — Train Loss Curves, Shortlisted  [{ACTIVE_FOLDER}]', fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
for variable in TARGET_VARIABLES:
    if variable not in baselines:
        continue
    all_exps = [baselines[variable]] + shortlists[variable]
    mae_col = f'{variable}_mae'
    vdf = df[df['experiment'].isin(all_exps) & df[mae_col].notna()]
    if len(vdf) == 0:
        continue

    exps_with_curves = [e for e in all_exps if e in vdf['experiment'].values]
    if not exps_with_curves:
        continue

    unit = '°C' if variable in ('tmax', 't2m') else 'm/s'
    n_exp = len(exps_with_curves)

    fig, axes = plt.subplots(2, n_exp, figsize=(5 * n_exp, 8), sharey='row')
    if n_exp == 1:
        axes = axes.reshape(-1, 1)

    for col, exp_name in enumerate(exps_with_curves):
        edata = vdf[vdf['experiment'] == exp_name]
        label = edata['label'].iloc[0]

        for _, row in edata.iterrows():
            curves = load_curves(row['run_dir'])
            if curves is None:
                continue
            seed_label = f's{row["seed"]}'

            if 'val_maes' in curves:
                val_maes = curves['val_maes']
                variables = row['variables']
                if isinstance(variables, str):
                    variables = [variables]
                if variable in variables:
                    vi = variables.index(variable)
                    if val_maes.ndim == 2 and vi < val_maes.shape[1]:
                        axes[0, col].plot(val_maes[:, vi], alpha=0.7, label=seed_label)
                    elif val_maes.ndim == 1:
                        axes[0, col].plot(val_maes, alpha=0.7, label=seed_label)

            if 'val_losses' in curves:
                axes[1, col].plot(curves['val_losses'], alpha=0.7, label=seed_label)

        axes[0, col].set_title(label, fontsize=8)
        axes[1, col].set_xlabel('Epoch')
        for row_idx in range(2):
            axes[row_idx, col].legend(fontsize=7)
            axes[row_idx, col].grid(alpha=0.3)

    axes[0, 0].set_ylabel(f'Val MAE ({unit})')
    axes[1, 0].set_ylabel('Val NLL')
    plt.suptitle(f'{variable} — Training Dynamics, Shortlisted  [{ACTIVE_FOLDER}]', fontsize=13)
    plt.tight_layout()
    plt.show()


## 8. Seasonal error patterns

How does performance vary across seasons? Temperature errors typically
peak in transition seasons (MAM, SON) while wind errors are highest in
winter (DJF) when synoptic variability is greatest.


In [ ]:
for variable in TARGET_VARIABLES:
    if variable not in baselines:
        continue
    all_exps = [baselines[variable]] + shortlists[variable]
    season_cols = [f'{variable}_mae_{s}' for s in ['DJF', 'MAM', 'JJA', 'SON']]
    has_seasonal = any(df[c].notna().any() for c in season_cols if c in df.columns)
    if not has_seasonal:
        continue

    experiments = [e for e in all_exps
                   if df[(df['experiment'] == e) & df[f'{variable}_mae'].notna()].shape[0] > 0]
    if not experiments:
        continue

    fig, ax = plt.subplots(figsize=(10, 5))
    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    x = np.arange(len(seasons))
    width = 0.8 / max(len(experiments), 1)

    for i, exp in enumerate(experiments):
        edata = df[df['experiment'] == exp]
        colour = edata['colour'].iloc[0]
        label = edata['label'].iloc[0]
        means = []
        for s in seasons:
            col = f'{variable}_mae_{s}'
            if col in edata.columns:
                vals = edata[col].dropna()
                means.append(vals.mean() if len(vals) > 0 else 0)
            else:
                means.append(0)
        ax.bar(x + i * width, means, width, label=label, color=colour, alpha=0.85)

    ax.set_xticks(x + width * len(experiments) / 2)
    ax.set_xticklabels(seasons)
    unit = '°C' if variable in ('tmax', 't2m') else 'm/s'
    ax.set_ylabel(f'MAE ({unit})')
    ax.set_title(f'{variable} — Seasonal MAE Breakdown, Shortlisted  [{ACTIVE_FOLDER}]')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.show()


## 9. Uncertainty calibration


In [ ]:
for variable in TARGET_VARIABLES:
    if variable not in baselines:
        continue
    all_exps = [baselines[variable]] + shortlists[variable]
    w1_col = f'{variable}_within_1sigma'
    w2_col = f'{variable}_within_2sigma'

    mask = df[w1_col].notna() if w1_col in df.columns else pd.Series(False, index=df.index)
    vdf = df[mask & df['experiment'].isin(all_exps)]
    if len(vdf) == 0:
        continue

    experiments = [e for e in all_exps if e in vdf['experiment'].values]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(experiments))

    for i, exp in enumerate(experiments):
        edata = vdf[vdf['experiment'] == exp]
        colour = edata['colour'].iloc[0]
        ax1.bar(i, edata[w1_col].mean() * 100, color=colour, alpha=0.85, edgecolor='black', linewidth=0.5)
        ax2.bar(i, edata[w2_col].mean() * 100, color=colour, alpha=0.85, edgecolor='black', linewidth=0.5)

    ax1.axhline(68.3, color='red', linestyle='--', label='Ideal (68.3%)')
    ax2.axhline(95.4, color='red', linestyle='--', label='Ideal (95.4%)')

    labels = [vdf[vdf['experiment'] == e]['label'].iloc[0][:25] for e in experiments]
    for ax, title in [(ax1, f'{variable} — Within 1σ'), (ax2, f'{variable} — Within 2σ')]:
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=8)
        ax.set_ylabel('%')
        ax.set_title(f'{title}  [{ACTIVE_FOLDER}]')
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    plt.show()


## 10. Per-station error analysis

Loads per-station errors from `test_station_errors.npz` (written by
`evaluate.py`). Skipped silently if those files don't exist for a run.


In [ ]:
def load_station_errors(run_dir):
    """Load per-station errors and metadata from a run directory."""
    p = Path(run_dir) / 'test_station_errors.npz'
    if not p.exists():
        return None
    return dict(np.load(p, allow_pickle=True))

In [ ]:
def plot_station_analysis(df, variable, shortlist):
    """Per-station error analysis for shortlisted experiments."""
    mae_col = f'{variable}_mae'
    vdf = df[df['experiment'].isin(shortlist) & df[mae_col].notna()]
    if len(vdf) == 0:
        return

    for exp_name in shortlist:
        edata = vdf[vdf['experiment'] == exp_name]
        if len(edata) == 0:
            continue
        row = edata.iloc[0]
        station_data = load_station_errors(row['run_dir'])
        if station_data is None:
            print(f'No station errors for {exp_name} — re-run evaluate.py')
            continue

        station_mae = station_data.get(f'{variable}_station_mae')
        station_count = station_data.get(f'{variable}_station_count')
        lats = station_data['station_lats']
        lons = station_data['station_lons']
        elevs = station_data['station_elevs']
        delta_elevs = station_data['station_delta_elevs']

        if station_mae is None:
            continue

        valid = station_count > 10
        if valid.sum() == 0:
            continue

        s_mae = station_mae[valid]
        s_lat = lats[valid]
        s_lon = lons[valid]
        s_elev = elevs[valid]
        s_delev = delta_elevs[valid]
        s_count = station_count[valid]

        unit = '°C' if variable == 'tmax' else 'm/s'
        label = row['label']
        colour = row['colour']

        # ---- Figure 1: Geographic map ----
        fig, ax = plt.subplots(figsize=(12, 7))
        vmin, vmax = np.percentile(s_mae, 5), np.percentile(s_mae, 95)
        sc = ax.scatter(s_lon, s_lat, c=s_mae, cmap='RdYlGn_r', s=20,
                        alpha=0.8, edgecolors='grey', linewidth=0.3,
                        vmin=vmin, vmax=vmax)
        cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.02)
        cbar.set_label(f'Station MAE ({unit})', fontsize=11)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_title(f'{variable} — Station MAE Geographic Distribution\n{label}',
                     fontsize=13)
        ax.set_aspect('equal')
        ax.grid(alpha=0.15)
        plt.show()

        # ---- Figure 2: MAE distribution + scatter vs terrain features ----
        fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

        ax = axes[0]
        ax.hist(s_mae, bins=40, color=colour, alpha=0.75, edgecolor='white',
                linewidth=0.5)
        ax.axvline(np.mean(s_mae), color='red', linestyle='--', linewidth=1.2,
                   label=f'Mean: {np.mean(s_mae):.3f}')
        ax.axvline(np.median(s_mae), color='darkorange', linestyle='--', linewidth=1.2,
                   label=f'Median: {np.median(s_mae):.3f}')
        ax.set_xlabel(f'Station MAE ({unit})')
        ax.set_ylabel('Count')
        ax.set_title('MAE Distribution')
        ax.legend(fontsize=8)

        ax = axes[1]
        ax.scatter(s_elev, s_mae, alpha=0.25, s=8, color=colour, rasterized=True)
        if len(s_elev) > 10:
            z = np.polyfit(s_elev, s_mae, 1)
            x_range = np.linspace(s_elev.min(), s_elev.max(), 100)
            ax.plot(x_range, np.poly1d(z)(x_range), 'r-', linewidth=1.5)
            corr = np.corrcoef(s_elev, s_mae)[0, 1]
            ax.text(0.95, 0.95, f'r = {corr:.3f}', transform=ax.transAxes,
                    ha='right', va='top', fontsize=10,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
        ax.set_xlabel('Elevation (m)')
        ax.set_ylabel(f'MAE ({unit})')
        ax.set_title('MAE vs Elevation')

        ax = axes[2]
        abs_delev = np.abs(s_delev)
        ax.scatter(abs_delev, s_mae, alpha=0.25, s=8, color=colour, rasterized=True)
        if len(s_delev) > 10:
            z = np.polyfit(abs_delev, s_mae, 1)
            x_range = np.linspace(abs_delev.min(), abs_delev.max(), 100)
            ax.plot(x_range, np.poly1d(z)(x_range), 'r-', linewidth=1.5)
            corr = np.corrcoef(abs_delev, s_mae)[0, 1]
            ax.text(0.95, 0.95, f'r = {corr:.3f}', transform=ax.transAxes,
                    ha='right', va='top', fontsize=10,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
        ax.set_xlabel('|Δ-Elevation| (m)')
        ax.set_ylabel(f'MAE ({unit})')
        ax.set_title('MAE vs |Δ-Elevation|')

        ax = axes[3]
        ax.scatter(s_lat, s_mae, alpha=0.25, s=8, color=colour, rasterized=True)
        if len(s_lat) > 10:
            corr = np.corrcoef(s_lat, s_mae)[0, 1]
            ax.text(0.95, 0.95, f'r = {corr:.3f}', transform=ax.transAxes,
                    ha='right', va='top', fontsize=10,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
        ax.set_xlabel('Latitude')
        ax.set_ylabel(f'MAE ({unit})')
        ax.set_title('MAE vs Latitude')

        plt.suptitle(f'{variable} — Error vs Station Characteristics: {label}',
                     fontsize=13, y=1.02)
        plt.tight_layout()
        plt.show()

        # ---- Figure 3: Worst and best stations ----
        sorted_idx = np.argsort(s_mae)
        worst_10 = sorted_idx[-10:][::-1]
        best_10 = sorted_idx[:10]

        rows = []
        for idx in worst_10:
            rows.append({
                'Rank': 'WORST',
                f'MAE ({unit})': f'{s_mae[idx]:.3f}',
                'Lat': f'{s_lat[idx]:.2f}',
                'Lon': f'{s_lon[idx]:.2f}',
                'Elev (m)': f'{s_elev[idx]:.0f}',
                'ΔElev (m)': f'{s_delev[idx]:.0f}',
                'N obs': f'{s_count[idx]:.0f}',
            })
        for idx in best_10:
            rows.append({
                'Rank': 'BEST',
                f'MAE ({unit})': f'{s_mae[idx]:.3f}',
                'Lat': f'{s_lat[idx]:.2f}',
                'Lon': f'{s_lon[idx]:.2f}',
                'Elev (m)': f'{s_elev[idx]:.0f}',
                'ΔElev (m)': f'{s_delev[idx]:.0f}',
                'N obs': f'{s_count[idx]:.0f}',
            })

        station_table = pd.DataFrame(rows)
        print(f'\n{variable} — {label}: Worst and Best Stations')
        print(station_table.to_string(index=False))

        print(f'\nSummary: {len(s_mae)} stations, '
              f'mean MAE={np.mean(s_mae):.3f}, median={np.median(s_mae):.3f}, '
              f'std={np.std(s_mae):.3f} {unit}')
        print(f'Correlations: elev r={np.corrcoef(s_elev, s_mae)[0,1]:.3f}, '
              f'|Δelev| r={np.corrcoef(np.abs(s_delev), s_mae)[0,1]:.3f}, '
              f'lat r={np.corrcoef(s_lat, s_mae)[0,1]:.3f}')


In [ ]:
for variable in TARGET_VARIABLES:
    if variable not in baselines:
        continue
    all_exps = [baselines[variable]] + shortlists[variable]
    plot_station_analysis(df, variable, all_exps)


## 11. Per-station baseline-vs-TESSERA comparison

Side-by-side maps of baseline vs the top TESSERA variant, plus a
delta map showing where TESSERA wins / loses.


In [ ]:
def compare_station_errors(df, variable, exp_baseline, exp_tessera):
    """Compare per-station errors between baseline and TESSERA experiments."""
    mae_col = f'{variable}_mae'

    bl_data = df[(df['experiment'] == exp_baseline) & df[mae_col].notna()]
    te_data = df[(df['experiment'] == exp_tessera) & df[mae_col].notna()]

    if len(bl_data) == 0 or len(te_data) == 0:
        print(f'Missing data for {exp_baseline} or {exp_tessera}')
        return

    bl_stations = load_station_errors(bl_data.iloc[0]['run_dir'])
    te_stations = load_station_errors(te_data.iloc[0]['run_dir'])

    if bl_stations is None or te_stations is None:
        print('No station error files — re-run evaluate.py')
        return

    bl_mae = bl_stations.get(f'{variable}_station_mae')
    te_mae = te_stations.get(f'{variable}_station_mae')
    bl_count = bl_stations.get(f'{variable}_station_count')
    te_count = te_stations.get(f'{variable}_station_count')

    if bl_mae is None or te_mae is None:
        return

    valid = (bl_count > 10) & (te_count > 10)
    if valid.sum() == 0:
        return

    delta_mae = bl_mae[valid] - te_mae[valid]
    lats = bl_stations['station_lats'][valid]
    lons = bl_stations['station_lons'][valid]
    elevs = bl_stations['station_elevs'][valid]
    delta_elevs = bl_stations['station_delta_elevs'][valid]

    unit = '°C' if variable == 'tmax' else 'm/s'

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    vmax = np.percentile(np.abs(delta_mae), 95)
    sc = ax.scatter(lons, lats, c=delta_mae, cmap='RdBu', s=15, alpha=0.7,
                    edgecolors='none', vmin=-vmax, vmax=vmax)
    plt.colorbar(sc, ax=ax, label=f'ΔMAE ({unit}, +ve = TESSERA better)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Geographic Pattern of TESSERA Improvement')
    ax.set_aspect('equal')

    ax = axes[0, 1]
    ax.hist(delta_mae, bins=50, color='steelblue', alpha=0.7, edgecolor='black',
            linewidth=0.3)
    ax.axvline(0, color='black', linewidth=1)
    ax.axvline(np.mean(delta_mae), color='red', linestyle='--',
               label=f'Mean: {np.mean(delta_mae):+.3f}')
    n_improved = (delta_mae > 0).sum()
    n_total = len(delta_mae)
    ax.set_title(f'ΔMAE Distribution ({n_improved}/{n_total} stations improved)')
    ax.set_xlabel(f'ΔMAE ({unit})')
    ax.legend()

    ax = axes[1, 0]
    ax.scatter(elevs, delta_mae, alpha=0.3, s=10)
    ax.axhline(0, color='black', linewidth=0.5)
    if len(elevs) > 10:
        corr = np.corrcoef(elevs, delta_mae)[0, 1]
        ax.set_title(f'Improvement vs Elevation (r={corr:.3f})')
    ax.set_xlabel('Elevation (m)')
    ax.set_ylabel(f'ΔMAE ({unit})')

    ax = axes[1, 1]
    abs_delev = np.abs(delta_elevs)
    ax.scatter(abs_delev, delta_mae, alpha=0.3, s=10)
    ax.axhline(0, color='black', linewidth=0.5)
    if len(abs_delev) > 10:
        corr = np.corrcoef(abs_delev, delta_mae)[0, 1]
        ax.set_title(f'Improvement vs |ΔElevation| (r={corr:.3f})')
    ax.set_xlabel('|Δ-Elevation| (m)')
    ax.set_ylabel(f'ΔMAE ({unit})')

    bl_label = bl_data.iloc[0]['label']
    te_label = te_data.iloc[0]['label']
    plt.suptitle(
        f'{variable} — Station-Level Comparison\n{te_label} vs {bl_label}',
        fontsize=13
    )
    plt.show()

    print(f'\n{variable} station-level improvement summary:')
    print(f'  Stations compared: {n_total}')
    print(f'  Stations improved: {n_improved} ({n_improved/n_total*100:.1f}%)')
    print(f'  Mean ΔMAE: {np.mean(delta_mae):+.4f} {unit}')
    print(f'  Median ΔMAE: {np.median(delta_mae):+.4f} {unit}')

    from scipy import stats as sp_stats
    t_stat, p_val = sp_stats.ttest_rel(bl_mae[valid], te_mae[valid])
    print(f'  Paired t-test: t={t_stat:.3f}, p={p_val:.4f}')
    w_stat, w_pval = sp_stats.wilcoxon(delta_mae)


In [ ]:
for variable in TARGET_VARIABLES:
    if variable not in baselines or not shortlists.get(variable):
        continue
    compare_station_errors(df, variable, baselines[variable], shortlists[variable][0])


## 12. Regional comparison

Per-region MAE breakdown using lat/lon bounding boxes. The default
`REGIONS` dict in this cell is tailored to Europe; for non-EU folders
you may want to redefine it (e.g. for `snapshot_14y_us`, regions like
"Rocky Mountains" or "Florida coast" would replace the European ones).
The function itself is region-agnostic — only the dict needs editing.


In [ ]:
# Define regions of interest by lat/lon bounding boxes.
REGIONS = {
    'Alps': {'lat_min': 45.5, 'lat_max': 48.0, 'lon_min': 6.0, 'lon_max': 16.0},
    'Norway': {'lat_min': 58.0, 'lat_max': 71.0, 'lon_min': 4.0, 'lon_max': 16.0},
    'Iberian Peninsula': {'lat_min': 36.0, 'lat_max': 43.5, 'lon_min': -10.0, 'lon_max': 3.0},
    'British Isles': {'lat_min': 50.0, 'lat_max': 59.0, 'lon_min': -11.0, 'lon_max': 2.0},
    'North European Plain': {'lat_min': 50.0, 'lat_max': 55.0, 'lon_min': 5.0, 'lon_max': 25.0},
    'Mediterranean Coast': {'lat_min': 36.0, 'lat_max': 44.0, 'lon_min': -5.0, 'lon_max': 20.0},
}

# Also define terrain complexity categories based on delta-elevation.
TERRAIN_CATEGORIES = {
    'Flat (|Δelev| < 50m)': lambda de: np.abs(de) < 50,
    'Moderate (50-200m)': lambda de: (np.abs(de) >= 50) & (np.abs(de) < 200),
    'Complex (200-500m)': lambda de: (np.abs(de) >= 200) & (np.abs(de) < 500),
    'Extreme (>500m)': lambda de: np.abs(de) >= 500,
}


def regional_comparison(df, variable, experiments_to_compare):
    """Compare experiments by geographic region and terrain complexity (seed-averaged).

    Each experiment is evaluated against ITS OWN station set — the VAE
    variants filter out stations without valid latents, so their station
    arrays have fewer rows than the baseline. Computing region_mask per
    experiment (not once against the baseline) avoids the index
    mismatch that would otherwise occur.
    """
    mae_col = f'{variable}_mae'
    unit = '°C' if variable in ('tmax', 't2m') else 'm/s'

    def load_experiment(exp_name):
        """Return (per_seed_station_maes, meta_dict, station_count) or Nones.

        meta_dict contains 'station_lats', 'station_lons',
        'station_delta_elevs' — the station arrays belonging to this
        experiment specifically.
        """
        edata = df[df['experiment'] == exp_name]
        all_maes = []
        meta = None
        s_count = None
        for _, row in edata.iterrows():
            sd = load_station_errors(row['run_dir'])
            if sd is None:
                continue
            if meta is None:
                meta = sd
                s_count = sd.get(f'{variable}_station_count')
            s_mae = sd.get(f'{variable}_station_mae')
            if s_mae is not None:
                all_maes.append(s_mae)
        if not all_maes or meta is None:
            return None, None, None
        return all_maes, meta, s_count

    region_results = []
    terrain_results = []

    for exp_name in experiments_to_compare:
        seed_maes, meta, s_count = load_experiment(exp_name)
        if seed_maes is None:
            continue

        edata = df[df['experiment'] == exp_name]
        label = edata['label'].iloc[0]

        # Per-experiment station arrays. Their lengths match this
        # experiment's seed_maes entries, which is what matters.
        lats = meta['station_lats']
        lons = meta['station_lons']
        delta_elevs = meta['station_delta_elevs']
        valid = s_count > 10 if s_count is not None else np.ones(len(lats), dtype=bool)

        # Regions.
        for region_name, bounds in REGIONS.items():
            region_mask = (
                valid
                & (lats >= bounds['lat_min']) & (lats <= bounds['lat_max'])
                & (lons >= bounds['lon_min']) & (lons <= bounds['lon_max'])
            )
            n_stations = int(region_mask.sum())
            if n_stations < 5:
                continue
            seed_means = [m[region_mask].mean() for m in seed_maes]
            rmean = np.mean(seed_means)
            rstd = np.std(seed_means) if len(seed_means) > 1 else 0.0
            region_results.append({
                'Experiment': label,
                'Region': region_name,
                f'MAE ({unit})': f'{rmean:.3f}±{rstd:.3f}',
                'Stations': n_stations,
            })

        # Terrain categories.
        for cat_name, cat_fn in TERRAIN_CATEGORIES.items():
            cat_mask = valid & cat_fn(delta_elevs)
            n_stations = int(cat_mask.sum())
            if n_stations < 5:
                continue
            seed_means = [m[cat_mask].mean() for m in seed_maes]
            tmean = np.mean(seed_means)
            tstd = np.std(seed_means) if len(seed_means) > 1 else 0.0
            terrain_results.append({
                'Experiment': label,
                'Terrain': cat_name,
                f'MAE ({unit})': f'{tmean:.3f}±{tstd:.3f}',
                'Stations': n_stations,
            })

    # Display regional results.
    if region_results:
        rdf = pd.DataFrame(region_results)
        pivot = rdf.pivot_table(
            index='Experiment', columns='Region',
            values=f'MAE ({unit})', aggfunc='first',
        )
        print(f'\n{"=" * 100}')
        print(f'{variable} — Regional MAE Comparison ({unit}, seed-averaged)')
        print(f'{"=" * 100}')
        print(pivot.to_string())

        fig, ax = plt.subplots(figsize=(16, 5))
        pivot_numeric = pivot.map(
            lambda x: float(x.split('±')[0]) if isinstance(x, str) else float(x)
        )
        pivot_numeric.plot(kind='bar', ax=ax, width=0.8)
        ax.set_ylabel(f'MAE ({unit})')
        ax.set_title(f'{variable} — MAE by Region (seed-averaged)')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
        plt.xticks(rotation=35, ha='right', fontsize=7)
        plt.tight_layout()
        plt.show()

    # Display terrain results.
    if terrain_results:
        tdf = pd.DataFrame(terrain_results)
        pivot_t = tdf.pivot_table(
            index='Experiment', columns='Terrain',
            values=f'MAE ({unit})', aggfunc='first',
        )
        print(f'\n{"=" * 100}')
        print(f'{variable} — MAE by Terrain Complexity ({unit}, seed-averaged)')
        print(f'{"=" * 100}')
        print(pivot_t.to_string())

        fig, ax = plt.subplots(figsize=(16, 5))
        pivot_t_numeric = pivot_t.map(
            lambda x: float(x.split('±')[0]) if isinstance(x, str) else float(x)
        )
        pivot_t_numeric.plot(kind='bar', ax=ax, width=0.8)
        ax.set_ylabel(f'MAE ({unit})')
        ax.set_title(f'{variable} — MAE by Terrain Complexity (seed-averaged)')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
        plt.xticks(rotation=35, ha='right', fontsize=7)
        plt.tight_layout()
        plt.show()


In [ ]:
def regional_comparison_with_density(df, variable, experiments_to_compare=None):
    """Compare experiments by region with station density context.

    Per-region comparisons intersect baseline and compared experiment
    station IDs so the subtraction is computed over the same physical
    stations in both models. (VAE variants filter out stations without
    valid latents, so their station arrays are shorter than the
    baseline's — without intersection we'd compare different station
    subsets in each region.)
    """
    mae_col = f'{variable}_mae'
    mask = df[mae_col].notna()
    if not mask.any():
        return

    if experiments_to_compare is None:
        experiments_to_compare = df[mask].sort_values(
            mae_col, key=lambda x: df.loc[x.index].groupby('experiment')[mae_col].transform('mean')
        )['experiment'].unique()[:6]

    unit = '°C' if variable in ('tmax', 't2m') else 'm/s'

    def load_avg_station_mae(exp_name):
        """Seed-averaged station MAE + metadata for an experiment.

        Returns (avg_mae, meta) where meta carries this experiment's own
        station arrays — lats/lons/delta_elevs/ids, whose length matches
        avg_mae. Also returns the station count array so callers can
        gate on sample sufficiency per station.
        """
        edata = df[df['experiment'] == exp_name]
        all_maes = []
        meta = None
        for _, row in edata.iterrows():
            sd = load_station_errors(row['run_dir'])
            if sd is None:
                continue
            if meta is None:
                meta = sd
            s_mae = sd.get(f'{variable}_station_mae')
            if s_mae is not None:
                all_maes.append(s_mae)
        if not all_maes or meta is None:
            return None, None
        return np.mean(all_maes, axis=0), meta

    # --- Baseline: load once, derive density/metadata tables from it. ---
    baseline_exp = experiments_to_compare[0]
    bl_mae, bl_meta = load_avg_station_mae(baseline_exp)
    if bl_mae is None:
        print('No station errors for baseline')
        return

    bl_lats = bl_meta['station_lats']
    bl_lons = bl_meta['station_lons']
    bl_delta_elevs = bl_meta['station_delta_elevs']
    bl_ids = bl_meta.get('station_ids')
    bl_count = bl_meta.get(f'{variable}_station_count')
    bl_valid = bl_count > 10 if bl_count is not None else np.ones(len(bl_lats), dtype=bool)

    # --- Region metadata table (from baseline stations). ---
    print(f'\n{"=" * 80}')
    print(f'Region Metadata')
    print(f'{"=" * 80}')
    print(f'{"Region":<25} {"Stations":>8} {"Area (deg²)":>12} {"Density":>12} {"Mean |Δelev|":>12}')
    print(f'{"-" * 80}')

    for region_name, bounds in REGIONS.items():
        region_mask = (
            bl_valid
            & (bl_lats >= bounds['lat_min']) & (bl_lats <= bounds['lat_max'])
            & (bl_lons >= bounds['lon_min']) & (bl_lons <= bounds['lon_max'])
        )
        n = int(region_mask.sum())
        area = (bounds['lat_max'] - bounds['lat_min']) * (bounds['lon_max'] - bounds['lon_min'])
        density = n / area if area > 0 else 0
        mean_delev = np.abs(bl_delta_elevs[region_mask]).mean() if n > 0 else 0.0
        print(f'{region_name:<25} {n:>8} {area:>12.1f} {density:>12.2f} {mean_delev:>12.1f}')

    # --- Per-experiment comparison against baseline. ---
    print(f'\n{"=" * 80}')
    print(f'{variable} — TESSERA Improvement vs Baseline by Region ({unit})')
    print(f'{"=" * 80}')

    for exp_name in experiments_to_compare[1:]:
        exp_mae, exp_meta = load_avg_station_mae(exp_name)
        if exp_mae is None:
            continue

        edata = df[df['experiment'] == exp_name]
        label = edata['label'].iloc[0]

        # Align baseline and experiment by station ID. If either side
        # doesn't carry station_ids, fall back to positional alignment
        # (and warn) — this is the legacy behaviour for older runs.
        exp_ids = exp_meta.get('station_ids')
        if bl_ids is not None and exp_ids is not None:
            exp_id_to_row = {sid: i for i, sid in enumerate(exp_ids)}
            # For each baseline station, the corresponding row in the
            # experiment arrays (or -1 if that station isn't in the exp).
            bl_to_exp = np.array(
                [exp_id_to_row.get(sid, -1) for sid in bl_ids], dtype=np.int64,
            )
            shared_mask = bl_to_exp >= 0
            if not shared_mask.any():
                print(f'\n  {label}: no shared stations with baseline, skipping')
                continue
            # exp_mae_aligned[i] = exp_mae value for the station at bl row i
            # (only valid where shared_mask is True).
            exp_mae_aligned = np.full(len(bl_ids), np.nan, dtype=np.float32)
            exp_mae_aligned[shared_mask] = exp_mae[bl_to_exp[shared_mask]]
        else:
            if len(exp_mae) != len(bl_mae):
                print(
                    f'\n  {label}: station_ids missing and lengths differ '
                    f'(bl={len(bl_mae)}, exp={len(exp_mae)}); skipping'
                )
                continue
            exp_mae_aligned = exp_mae
            shared_mask = np.ones(len(bl_mae), dtype=bool)

        print(f'\n  {label}:')
        for region_name, bounds in REGIONS.items():
            region_mask = (
                bl_valid
                & shared_mask
                & (bl_lats >= bounds['lat_min']) & (bl_lats <= bounds['lat_max'])
                & (bl_lons >= bounds['lon_min']) & (bl_lons <= bounds['lon_max'])
            )
            n = int(region_mask.sum())
            if n < 5:
                continue
            bl_regional = bl_mae[region_mask].mean()
            exp_regional = exp_mae_aligned[region_mask].mean()
            delta = bl_regional - exp_regional
            pct = delta / bl_regional * 100
            direction = '▲' if delta > 0 else '▼'
            area = (bounds['lat_max'] - bounds['lat_min']) * (bounds['lon_max'] - bounds['lon_min'])
            print(f'    {region_name:<22} {direction} {delta:+.3f} ({pct:+.1f}%)  '
                  f'[n={n}, density={n/area:.1f} st/deg²]')


In [ ]:
for variable in TARGET_VARIABLES:
    if variable not in baselines:
        continue
    all_exps = [baselines[variable]] + shortlists[variable][:2]  # baseline + top 2
    if len(all_exps) < 2:
        continue
    regional_comparison_with_density(df, variable, experiments_to_compare=all_exps)
